In [73]:
import pandas as pd 
import numpy as np 

In [74]:
# load all data
train = pd.read_csv('../data/preprocecssed/train.csv')
labels = pd.read_csv('../data/raw/train_labels.csv')
test = pd.read_csv('../data/raw/test_values.csv')


In [75]:
ordinal_categories = {
    'land_surface_condition': ['t', 'n', 'o'],
    'foundation_type': ['i', 'w', 'u', 'h', 'r'],
    'roof_type': ['x', 'n', 'q'],
    'ground_floor_type': ['v', 'm', 'z', 'x', 'f'],
    'other_floor_type': ['s', 'j', 'x', 'q'],
    'position': ['j', 'o', 's', 't'],
    'plan_configuration': ['c', 'a', 'o', 'm', 'u', 's', 'n', 'd', 'q', 'f'],
}

ord_cols = list(ordinal_categories.keys())
ord_cats = [ordinal_categories[c] for c in ord_cols]

ordinal_encoder = OrdinalEncoder(
    categories=ord_cats,
    handle_unknown="use_encoded_value",
    unknown_value=-1
)

In [76]:
train

,Unnamed: 0,building_id,geo_level_1_id,geo_level_2_id,geo_level_3_id,count_floors_pre_eq,age,area_percentage,height_percentage,land_surface_condition,...,area_per_floor,slenderness,volume_proxy,is_tall,old_mud_stone,families_per_floor,families_per_area,has_multiple_families,vertical_risk,damage_grade
0,0,802906,6,487,12198,2,30,6,5,t,...,3.000000,0.833333,30,0,1,0.500000,0.166667,0,10,3
1,1,28830,8,900,2812,2,10,8,7,o,...,4.000000,0.875000,56,0,0,0.500000,0.125000,0,14,2
2,2,94947,21,363,8973,2,10,5,5,t,...,2.500000,1.000000,25,0,0,0.500000,0.200000,0,10,3
3,3,590882,22,418,10694,2,10,6,5,t,...,3.000000,0.833333,30,0,0,0.500000,0.166667,0,10,2
4,4,201944,11,131,1488,3,30,8,9,t,...,2.666667,1.125000,72,0,0,0.333333,0.125000,0,27,3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
260596,260596,688636,25,1335,1621,1,55,6,3,n,...,6.000000,0.500000,18,0,1,1.000000,0.166667,0,3,2
260597,260597,669485,17,715,2060,2,0,6,5,t,...,3.000000,0.833333,30,0,0,0.500000,0.166667,0,10,3
260598,260598,602512,17,51,8163,3,55,6,7,t,...,2.000000,1.166667,42,0,1,0.333333,0.166667,0,21,3
260599,260599,151409,26,39,1851,2,10,14,6,t,...,7.000000,0.428571,84,0,0,0.500000,0.071429,0,12,2


In [77]:
X =   train.drop(['Unnamed: 0','damage_grade'], axis=1)
y = train['damage_grade']

In [78]:
X = pd.DataFrame(X)

In [79]:
num_cols = train.select_dtypes(include=['int64', 'float64']).columns.tolist()

# remove target if present
num_cols.remove('damage_grade')

# ordinal columns already defined
ord_cols = list(ordinal_categories.keys())

# nominal categorical = all object columns minus ordinal ones
nominal_cols = [
    c for c in X.select_dtypes(include='object').columns
    if c not in ord_cols
]

In [80]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('ord', ordinal_encoder, ord_cols),
        ('nom', OneHotEncoder(handle_unknown='ignore'), nominal_cols)
    ]
)

In [82]:
X

,building_id,geo_level_1_id,geo_level_2_id,geo_level_3_id,count_floors_pre_eq,age,area_percentage,height_percentage,land_surface_condition,foundation_type,...,height_per_floor,area_per_floor,slenderness,volume_proxy,is_tall,old_mud_stone,families_per_floor,families_per_area,has_multiple_families,vertical_risk
0,802906,6,487,12198,2,30,6,5,t,r,...,2.500000,3.000000,0.833333,30,0,1,0.500000,0.166667,0,10
1,28830,8,900,2812,2,10,8,7,o,r,...,3.500000,4.000000,0.875000,56,0,0,0.500000,0.125000,0,14
2,94947,21,363,8973,2,10,5,5,t,r,...,2.500000,2.500000,1.000000,25,0,0,0.500000,0.200000,0,10
3,590882,22,418,10694,2,10,6,5,t,r,...,2.500000,3.000000,0.833333,30,0,0,0.500000,0.166667,0,10
4,201944,11,131,1488,3,30,8,9,t,r,...,3.000000,2.666667,1.125000,72,0,0,0.333333,0.125000,0,27
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
260596,688636,25,1335,1621,1,55,6,3,n,r,...,3.000000,6.000000,0.500000,18,0,1,1.000000,0.166667,0,3
260597,669485,17,715,2060,2,0,6,5,t,r,...,2.500000,3.000000,0.833333,30,0,0,0.500000,0.166667,0,10
260598,602512,17,51,8163,3,55,6,7,t,r,...,2.333333,2.000000,1.166667,42,0,1,0.333333,0.166667,0,21
260599,151409,26,39,1851,2,10,14,6,t,r,...,3.000000,7.000000,0.428571,84,0,0,0.500000,0.071429,0,12
